# Lead Scoring on UCI Bank Marketing DataThis notebook walks through building a lead scoring model end to end. The goal is to assign each prospective customer a 0-100 score that predicts how likely they are to convert, so a sales team can prioritize outreach.The methodology mirrors what I built at F Street Capital for hard money loan applications, rebuilt here on the public UCI Bank Marketing dataset.**Pipeline overview**1. Load and explore the data2. Engineer business-meaningful features3. Train a logistic regression classifier4. Convert probabilities into a 0-100 lead score5. Validate with AUC, tier analysis, and decile lift6. Inspect feature importances

In [ ]:
import syssys.path.insert(0, '../src')import pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom feature_engineering import engineer_features, get_categorical_features, get_numerical_featuresfrom model import train_model, get_feature_importancefrom scoring import score_leads, success_rate_by_tier, success_rate_by_decilefrom visualizations import (    plot_score_distribution, plot_roc_curve,    plot_success_by_tier, plot_feature_importance)pd.set_option('display.max_columns', 30)%matplotlib inline

## 1. Load and exploreThe UCI Bank Marketing dataset has 45,211 rows from a Portuguese bank's direct marketing campaigns. The target is whether a contacted customer subscribed to a term deposit.

In [ ]:
df = pd.read_csv('../data/bank-full.csv', sep=';')print(f"Shape: {df.shape}")df.head()

In [ ]:
print("Target distribution:")print(df['y'].value_counts(normalize=True).round(3))print(f"\nBaseline conversion rate: {(df['y'] == 'yes').mean():.1%}")

The baseline conversion rate is about 11.7 percent. This is the rate we'd see if we contacted everyone randomly. Any useful model needs to identify segments with conversion rates meaningfully above this baseline.

## 2. Feature engineeringRaw features are turned into business-meaningful buckets. For example, account balance becomes 5 tiers, age becomes 5 brackets, and campaign + previous contact counts get combined into a single 'contact intensity' feature.This bucketing approach makes the model output explainable to non-technical stakeholders and lets the same logic ship to a CRM workflow.

In [ ]:
df_eng = engineer_features(df)print("Engineered features:")for col in ['age_bucket', 'balance_bucket', 'contact_intensity', 'duration_bucket', 'prior_outcome']:    print(f"\n{col}:")    print(df_eng[col].value_counts())

## 3. Train the modelLogistic regression with class balancing, fit on a stratified 80/20 split. We use cross-validation to make sure the AUC isn't a single lucky split.

In [ ]:
pipeline, X_test, y_test, y_pred_proba, metrics = train_model(df)print(f"Test AUC: {metrics['test_auc']:.3f}")print(f"5-fold CV AUC: {metrics['cv_mean_auc']:.3f} +/- {metrics['cv_std_auc']:.3f}")print(f"Baseline rate: {metrics['baseline_rate']:.1%}")

## 4. Convert to lead scoresThe model output is a probability between 0 and 1. We map it to a 0-100 score and assign each lead to a tier (Hot, Warm, Lukewarm, Cold, Frozen).

In [ ]:
scored = score_leads(y_pred_proba, y_test.values)scored.head(10)

In [ ]:
tier_summary = success_rate_by_tier(scored)tier_summary

Conversion rate climbs sharply with score tier. The Hot tier converts at multiples of the baseline rate, which is exactly what a sales team needs in order to prioritize.

## 5. Decile liftDecile analysis shows whether the model produces a clean ranking. If the top decile converts much more than the bottom decile, the model is doing its job.

In [ ]:
decile_summary = success_rate_by_decile(scored)decile_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))ax.bar(decile_summary['decile'], decile_summary['success_rate'], color='#4a7ab8', edgecolor='white')ax.set_xlabel('Score Decile (1 = lowest, 10 = highest)')ax.set_ylabel('Conversion Rate (%)')ax.set_title('Conversion Rate by Score Decile')ax.grid(True, alpha=0.3, axis='y')ax.spines['top'].set_visible(False)ax.spines['right'].set_visible(False)plt.tight_layout()plt.show()

## 6. Feature importanceThe logistic regression coefficients tell us which features push a lead toward conversion (positive) versus away from it (negative).

In [ ]:
importance_df = get_feature_importance(pipeline, top_n=15)importance_df

In [ ]:
plot_feature_importance(importance_df, '../outputs/feature_importance_nb.png', top_n=15)from IPython.display import ImageImage('../outputs/feature_importance_nb.png')

## 7. ROC curve

In [ ]:
plot_roc_curve(y_test, y_pred_proba, '../outputs/roc_curve_nb.png')Image('../outputs/roc_curve_nb.png')

## Summary- Trained logistic regression on 45K leads with engineered features- Achieved AUC around 0.90 on the test set with stable CV scores- The top tier (Hot, 80+) converts at roughly 5-10x the baseline rate- Top features align with marketing intuition: prior campaign success, call duration, and contact type matter mostThis methodology is the same approach I used at F Street Capital for hard money loan applications, where the model achieved AUC 0.84 on 3,000+ applications and was deployed in HubSpot as a 0-100 scoring framework.